# Agent 8 — Capstone: Run Your Own Question

The whole pipeline in one place: decompose → gather → claims → verify →
report. Run it on the mini-web keyless, or on your own question with the
class key — including real web search via the server-side tool.

**Before anything runs:** what you send to a model leaves your machine.
Research questions about other people's private situations are theirs, not
yours. And the agent's report is a draft for a human — verify before you
repeat it.

In [ ]:
# The mini-web: nine pages about the (fictional) Riverside Community Garden.
# Small enough to read whole, real enough to research. One page is wrong on purpose.
MINIWEB = {
 "riverside-garden.org/about": {"date": "2026-05-10", "title": "Our garden today",
  "text": "The Riverside Community Garden has 60 plots and 48 member families. "
          "We grow vegetables for members and donate surplus to the food pantry."},
 "riverside-garden.org/history": {"date": "2023-05-02", "title": "Our history",
  "text": "Founded in 2019 with a dozen beds. The sign by the gate lists 48 plots, "
          "painted when we finished the 2023 season."},
 "riverside-garden.org/join": {"date": "2026-06-01", "title": "Join us",
  "text": "Want a plot? The waitlist currently holds 22 families. Members pay a "
          "small annual fee and share watering duties."},
 "lakeview-news.com/garden-expands": {"date": "2026-04-20", "title": "Garden adds 12 plots",
  "text": "The Riverside Community Garden completed its expansion this spring, "
          "taking the garden from 48 plots to 60. Organizers credit a city grant."},
 "lakeview-news.com/roundup-2023": {"date": "2023-09-15", "title": "Community roundup",
  "text": "At the Riverside garden, 31 member families closed out the 2023 season "
          "with a harvest festival."},
 "cityparks.gov/report-2026": {"date": "2026-03-14", "title": "Community garden census",
  "text": "Riverside Community Garden: 60 plots, 48 member families, established "
          "2019. Census conducted March 2026."},
 "cityparks.gov/grants-2025": {"date": "2025-11-08", "title": "2025 grant awards",
  "text": "Riverside Community Garden: $15,000 for expansion. The site's land "
          "lease with the parks department runs through 2028."},
 "gardenblog.example.com/visit": {"date": "2026-02-02", "title": "A visit to Riverside",
  "text": "Lovely afternoon at Riverside! I heard they have 600 plots now, which "
          "explains the crowds. The tomatoes were spectacular."},
 "gardenblog.example.com/opinion": {"date": "2026-01-05", "title": "Why gardens matter",
  "text": "Community gardens are the beating heart of a neighborhood. Riverside "
          "is a treasure and everyone loves it."},
}

import re as _re, collections as _c
def _words(text):
    return set(w for w in _re.findall(r"[a-z0-9]+", text.lower()) if len(w) > 2)
_DF = _c.Counter()                       # in how many pages does each word appear?
for _p in MINIWEB.values():
    for _w in _words(_p["title"] + " " + _p["text"]):
        _DF[_w] += 1

def search(query):
    """Score pages by shared words, each weighted by rarity (1/pages-containing-it).
    'riverside' is on every page and says nothing; 'waitlist' is on one and says a lot."""
    qwords = _words(query)
    scored = []
    for url, page in MINIWEB.items():
        shared = qwords & _words(page["title"] + " " + page["text"])
        scored.append((sum(1.0 / _DF[w] for w in shared), url, page["title"]))
    scored.sort(reverse=True)
    return [(url, title) for score, url, title in scored[:3] if score > 0.3]

def fetch(url):
    """Return a page's text with its receipt (url and date) attached."""
    page = MINIWEB[url]
    return {"url": url, "date": page["date"], "text": page["text"]}

print(f"{len(MINIWEB)} pages online.")
print("search('riverside garden plots') ->")
for url, title in search("riverside garden plots"):
    print("  ", url, "-", title)

In [ ]:
%pip install -q anthropic

In [ ]:
import os, getpass
# Ask your teacher for the class API key. It is never typed into a cell,
# never saved in the notebook - getpass keeps it out of your file.
try:
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Class API key: ")
    HAVE_KEY = len(os.environ["ANTHROPIC_API_KEY"]) > 10
except Exception:
    HAVE_KEY = False
print("Key loaded." if HAVE_KEY else "No key - the notebook still teaches: precomputed outputs are shown below each live cell.")

In [ ]:
MODEL = "claude-opus-5"

def ask(prompt, system=None, max_tokens=1000):
    """One model call, plain text in and out."""
    import anthropic
    client = anthropic.Anthropic()
    kwargs = dict(model=MODEL, max_tokens=max_tokens,
                  messages=[{"role": "user", "content": prompt}])
    if system:
        kwargs["system"] = system
    return client.messages.create(**kwargs).content[-1].text

def get_json(prompt, tries=3):
    """Ask for JSON only; parse; re-ask on failure. The retry pattern from Build with LLMs."""
    import json as _json
    for attempt in range(tries):
        text = ask(prompt + "\n\nReply with ONLY valid JSON.")
        try:
            start = text.index("[") if "[" in text.split("{")[0] else text.index("{")
            return _json.loads(text[start:])
        except (ValueError, KeyError):
            continue
    raise RuntimeError("no valid JSON after retries")

## The pipeline, assembled from lessons 3–7

In [ ]:
def run_research(question, angles):
    """The full offline flow over the mini-web. Each stage is a lesson."""
    # gather (lesson 4)
    seen, notes = set(), []
    for angle in angles:
        for url, _t in search(angle):
            if url in seen: continue
            seen.add(url)
            page = fetch(url)
            for s in page["text"].split(". "):
                if any(ch.isdigit() for ch in s):
                    notes.append({"note": s.strip().rstrip("."), "source": url, "date": page["date"]})
    # claims + verify (lessons 5-6, compressed: numbers + source counts)
    import re, collections
    support = collections.defaultdict(set)
    for n in notes:
        for num in re.findall(r"\$?[\d,]+", n["note"]):
            support[num].add(n["source"])
    claims, quarantine = [], []
    for num, sources in sorted(support.items()):
        row = {"value": num, "sources": sorted(sources), "n": len(sources)}
        (claims if len(sources) >= 1 else quarantine).append(row)
    # the 600-vs-60 style check: a value contradicted by a better-supported value
    for row in list(claims):
        for other in claims:
            if other["n"] > row["n"] and other["value"] != row["value"] and \
               row["n"] == 1 and other["n"] >= 2 and len(row["value"]) == len(other["value"]) + 1 \
               and row["value"].startswith(other["value"]):
                claims.remove(row)
                quarantine.append({**row, "reason": f"contradicted by {other['value']} ({other['n']} sources)"})
                break
    return notes, claims, quarantine

QUESTION = "Is the Riverside Community Garden growing?"
ANGLES = ["history of plot numbers at the Riverside garden",
          "Riverside garden member families census",
          "city grant funding for the Riverside garden expansion",
          "waitlist to join the Riverside garden",
          "a blog post about a visit to the Riverside garden"]

notes, claims, quarantine = run_research(QUESTION, ANGLES)
print(f"{len(notes)} notes, {len(claims)} claim values, {len(quarantine)} quarantined")
print("quarantine:", quarantine)
assert any(q.get("value") == "600" for q in quarantine), "the seeded typo should land in quarantine"

## The deliverables, written to files

In [ ]:
import csv, json as _json, pathlib

out = pathlib.Path("research_run")
out.mkdir(exist_ok=True)

(out / "run_report.md").write_text(f"""# Research run
Question: {QUESTION}
Angles: {ANGLES}
Pages fetched: {len({n['source'] for n in notes})} - Notes: {len(notes)}
Claim values: {len(claims)} - Quarantined: {len(quarantine)}
Trust note: multi-source values are strongest; single-source values need a
human to follow the receipt before repeating them.
""")
with (out / "claims.csv").open("w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["value", "n", "sources"])
    w.writeheader()
    for c in claims:
        w.writerow({"value": c["value"], "n": c["n"], "sources": ";".join(c["sources"])})
(out / "quarantine.json").write_text(_json.dumps(quarantine, indent=1))
print("wrote:", sorted(p.name for p in out.iterdir()))

## Real web search — the server-side tool

With the class key, the search tool can be the real web: include the
web-search tool in the request and the model searches during its turn,
returning text with the URLs it used. The loop's shape doesn't change —
search is still a requested tool; it just executes on the provider's side.

In [ ]:
if HAVE_KEY:
    import anthropic
    client = anthropic.Anthropic()
    resp = client.messages.create(
        model=MODEL, max_tokens=1200,
        tools=[{"type": "web_search_20260209", "name": "web_search", "max_uses": 3}],
        messages=[{"role": "user", "content":
                   "What did the James Webb telescope launch cost? Cite your sources."}])
    for block in resp.content:
        if block.type == "text":
            print(block.text)
else:
    print("Precomputed sample (live web search, 2 searches used):")
    print("  'NASA puts the James Webb Space Telescope's total development cost")
    print("   at about $10 billion [nasa.gov; planetary.org]...'")
    print("The response arrives with citations attached - and everything you")
    print("built still applies: those citations deserve the lesson-6 treatment")
    print("before you repeat the number.")

## Ship it

Your capstone ships four things (details on the lesson page):

1. **The report** — cited, hedged where single-source, with a Limits section.
2. **The claim table** — value, sources, verdict.
3. **The quarantine** — what was refused, and why.
4. **The run report** — angles, fetch counts, and your own trust paragraph:
   what you'd repeat from this report, and what you'd verify by hand first.

Pick a question with stakes — one where you care whether the answer is
right. Then give the report to the person deciding, and watch what they
check first.